In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"C:\Users\sudhanshu\Downloads\archive\PS_20174392719_1491204439457_log.csv")

In [3]:
df.shape

(6362620, 11)

In [4]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


## **DATA CLEANING**

In [6]:
df.isna().sum()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

## **EXPLORATORY DATA ANALYSIS**

In [7]:
df['type'].value_counts()

type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

In [8]:
df['isFraud'].value_counts()

isFraud
0    6354407
1       8213
Name: count, dtype: int64

In [9]:
df['isFraud'].mean()*100

np.float64(0.12908204481801522)

In [10]:
df.groupby('type')['amount'].agg(['count', 'sum', 'mean'])

,count,sum,mean
type,,,
CASH_IN,1399284,2.363674e+11,168920.242004
CASH_OUT,2237500,3.944130e+11,176273.964346
DEBIT,41432,2.271992e+08,5483.665314
PAYMENT,2151495,2.809337e+10,13057.604660
TRANSFER,532909,4.852920e+11,910647.009645


In [11]:
df.groupby('type')['isFraud'].sum()

type
CASH_IN        0
CASH_OUT    4116
DEBIT          0
PAYMENT        0
TRANSFER    4097
Name: isFraud, dtype: int64

In [12]:
df['balance_error'] = df['oldbalanceOrg'] - df['amount'] - df['newbalanceOrig']
df['balance_error'].describe()

count    6.362620e+06
mean    -2.010925e+05
std      6.066505e+05
min     -9.244552e+07
25%     -2.496411e+05
50%     -6.867726e+04
75%     -2.954230e+03
max      1.000000e-02
Name: balance_error, dtype: float64

In [13]:
df[(df['oldbalanceOrg'] == 0) & (df['newbalanceOrig'] == 0) & (df['amount'] > 0)].shape

(2088969, 12)

In [14]:
df['zero_balance_issue'] = ((df['oldbalanceOrg'] == 0) & (df['newbalanceOrig'] == 0) & (df['amount'] > 0)).astype(int)

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 13 columns):
 #   Column              Dtype  
---  ------              -----  
 0   step                int64  
 1   type                object 
 2   amount              float64
 3   nameOrig            object 
 4   oldbalanceOrg       float64
 5   newbalanceOrig      float64
 6   nameDest            object 
 7   oldbalanceDest      float64
 8   newbalanceDest      float64
 9   isFraud             int64  
 10  isFlaggedFraud      int64  
 11  balance_error       float64
 12  zero_balance_issue  int64  
dtypes: float64(6), int64(4), object(3)
memory usage: 631.1+ MB


In [16]:
df.groupby('zero_balance_issue')['isFraud'].mean()*100

zero_balance_issue
0    0.191593
1    0.001197
Name: isFraud, dtype: float64

In [17]:
df['emptied_account'] = ((df['oldbalanceOrg'] > 0) & (df['newbalanceOrig'] == 0) & (df['amount'] >= df['oldbalanceOrg'])).astype(int)

In [18]:
df.groupby('emptied_account')['isFraud'].mean()*100

emptied_account
0    0.004151
1    0.526904
Name: isFraud, dtype: float64

In [19]:
pd.crosstab(['type'], ['emptied_account'])

col_0,emptied_account
row_0,
type,1


In [20]:
df.groupby(['type','emptied_account'])['isFraud'].mean()

type      emptied_account
CASH_IN   0                  0.000000
CASH_OUT  0                  0.000030
          1                  0.004251
DEBIT     0                  0.000000
          1                  0.000000
PAYMENT   0                  0.000000
          1                  0.000000
TRANSFER  0                  0.000536
          1                  0.017204
Name: isFraud, dtype: float64

In [21]:
df[(df['emptied_account'] == 1) & (df['isFraud'] == 1)].sort_values('amount',ascending=False).head(10)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,balance_error,zero_balance_issue,emptied_account
6362583,741,CASH_OUT,10000000.0,C1668034607,10000000.0,0.0,C1250722530,192912.98,10192912.98,1,0,0.0,0,1
6362581,741,CASH_OUT,10000000.0,C677394894,10000000.0,0.0,C1866259073,0.00,10000000.00,1,0,0.0,0,1
6362579,741,CASH_OUT,10000000.0,C1647199421,10000000.0,0.0,C2077145577,35922.97,10035922.97,1,0,0.0,0,1
6362577,741,CASH_OUT,10000000.0,C421958672,10000000.0,0.0,C2034016962,0.00,10000000.00,1,0,0.0,0,1
4441,4,CASH_OUT,10000000.0,C351297720,10000000.0,0.0,C766681183,0.00,9941904.21,1,0,0.0,0,1
3960329,296,CASH_OUT,10000000.0,C1814200711,10000000.0,0.0,C270577353,920463.99,10920463.99,1,0,0.0,0,1
6361722,718,CASH_OUT,10000000.0,C1049094143,10000000.0,0.0,C1023840891,2517892.18,12517892.18,1,0,0.0,0,1
6362507,734,CASH_OUT,10000000.0,C131420902,10000000.0,0.0,C635621973,2571402.25,12571402.25,1,0,0.0,0,1
6362459,730,CASH_OUT,10000000.0,C2065498878,10000000.0,0.0,C502437065,819130.39,10819130.39,1,0,0.0,0,1
6362457,730,CASH_OUT,10000000.0,C315149365,10000000.0,0.0,C574552283,62895.13,10062895.13,1,0,0.0,0,1


## **Feature Engineering**

In [22]:
df['risk_score'] = ((df['emptied_account'] * 3) + (df['type'].isin(['TRANSFER', 'CASH_OUT ']).astype(int) * 2) + 
                   (df['amount'] > 200000).astype(int))

In [23]:
df.value_counts('type')

type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

In [24]:
df['hour'] = df['step'] % 24
df['high_value_tx'] = (df['amount'] > 200000).astype(int)
df['high_risk_type'] = df['type'].isin(['TRANSFER','CASH_OUT']).astype(int)

In [25]:
df.groupby('risk_score')['isFraud'].mean()

risk_score
0    0.000007
1    0.000013
2    0.000000
3    0.001389
4    0.006526
5    0.034735
6    0.013592
Name: isFraud, dtype: float64

In [26]:
df.columns

Index(['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig',
       'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud',
       'isFlaggedFraud', 'balance_error', 'zero_balance_issue',
       'emptied_account', 'risk_score', 'hour', 'high_value_tx',
       'high_risk_type'],
      dtype='object')

In [27]:
df.isnull().sum()

step                  0
type                  0
amount                0
nameOrig              0
oldbalanceOrg         0
newbalanceOrig        0
nameDest              0
oldbalanceDest        0
newbalanceDest        0
isFraud               0
isFlaggedFraud        0
balance_error         0
zero_balance_issue    0
emptied_account       0
risk_score            0
hour                  0
high_value_tx         0
high_risk_type        0
dtype: int64

## **IT'S TAKING TOO MUCH TIME BEACUASE WE HAVE 6M+ ROWS**

In [28]:
# from sqlalchemy import create_engine

# username = 'postgres'
# password = '25012004'
# host = 'localhost'
# port = '5433'
# database = 'fintech_databse'

# engine = create_engine(f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}")

# table_name = 'fintech_dataset'
# df.to_sql(table_name, engine, if_exists='replace', index=False)

# print(f"DF succesfully loaded in table '{table_name}' in database '{database}'.")

## **THERE FOR WE ARE USING THIS METHOD**

In [30]:
df.columns = df.columns.str.lower()

In [31]:
df.columns

Index(['step', 'type', 'amount', 'nameorig', 'oldbalanceorg', 'newbalanceorig',
       'namedest', 'oldbalancedest', 'newbalancedest', 'isfraud',
       'isflaggedfraud', 'balance_error', 'zero_balance_issue',
       'emptied_account', 'risk_score', 'hour', 'high_value_tx',
       'high_risk_type'],
      dtype='object')

In [32]:
df.to_csv("fintech_clean_data.csv", index=False)

In [33]:
from sqlalchemy import create_engine

username = 'postgres'
password = '25012004'
host = 'localhost'
port = '5433'
database = 'fintech_databse'

engine = create_engine(f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}")

table_name = 'fintech_dataset'
df.head(0).to_sql(table_name, engine, if_exists='replace', index=False)

0

In [34]:
import psycopg2

conn = psycopg2.connect(
    host = "localhost",
    port = "5433",
    database = "fintech_databse",
    user = "postgres",
    password = "25012004"
)

cursor = conn.cursor()

with open("fintech_clean_data.csv","r") as f:
    cursor.copy_expert("COPY fintech_dataset FROM STDIN WITH CSV HEADER",f)

conn.commit()
cursor.close()
conn.close()